# S6E10 | Subtract to Win: Negative-Weight Rank Blend
Collects every top public submission file + my own models, prints rank correlations, writes candidate blends (rank space, incl. negative-weight probes).

## Credits

- [PS-s6e10 | h-blend (pure.rank's)](https://www.kaggle.com/code/nina2025/ps-s6e10-h-blend-pure-rank-s) by nina2025 (built on work by Vladimir Demidov)
- [S6E10 TFM OOF and Test Predictions](https://www.kaggle.com/datasets/cdeotte/s6e10-tfm-oof-and-test-predictions) by Chris Deotte
- [S6E10 | Meta Stack on Public OOF | LB 0.96213](https://www.kaggle.com/code/goodpjw2008/s6e10-meta-stack-on-public-oof-lb-0-96213) and [S6E10 | Twin Stacks + Family Reg](https://www.kaggle.com/code/goodpjw2008/s6e10-twin-stacks-family-reg-lb-0-96187) by goodpjw2008
- [Summit Composer | 0.96210](https://www.kaggle.com/code/mayankraaz/s6e10-summit-composer-0-96210) by mayankraaz
- [On the journey to your first top 100](https://www.kaggle.com/code/daoviet/s6e10-on-the-journey-to-your-first-top-100) by daoviet
- [S6E10 | LB 0.96209 | Rank: 10](https://www.kaggle.com/code/rabari9999/s6e10-lb-0-96209-rank-10) by rabari9999
- [Secrets to Satisfaction (It's Stacking!)](https://www.kaggle.com/code/yeashusemwal/secrets-to-satisfaction-it-s-stacking) by Prabhav Semwal
- [[1] Airline S6E10 TabPFN](https://www.kaggle.com/code/mehrankazeminia/1-airline-s6e10-tabpfn) and [[3] Airline S6E10 Ensembling](https://www.kaggle.com/code/mehrankazeminia/3-airline-s6e10-ensembling) by Mehran Kazeminia
- [S6E10 | Are You Satisfied](https://www.kaggle.com/code/anhadmahajan06/s6e10-are-you-satisfied) by anhadmahajan06
- [Binary Aviation Satisfaction 129k](https://www.kaggle.com/datasets/arseniyshutko/binary-aviation-satisfaction-129k) by Arseniy Shutko

In [ ]:
import glob, os, json, re
import numpy as np, pandas as pd
from scipy.stats import rankdata

INP = '/kaggle/input' if os.path.isdir('/kaggle/input') else os.environ.get('BLEND_INPUT', './input')
OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else './working'
os.makedirs(OUT, exist_ok=True)

comp = [d for d in glob.glob(INP + '/**/playground-series-s6e10', recursive=True) if os.path.isdir(d)][0]
te = pd.read_csv(comp + '/test.csv', usecols=['id'])
IDS = te['id'].values
N = len(IDS)

def short(path):
    p = path[len(INP):].strip('/').split('/')
    keep = [x for x in p if x not in ('notebooks', 'datasets', 'competitions', 'kaggle', 'working', 'input')]
    return '/'.join(keep[-3:])

C = {}
# 1) every submission-format csv: id + one numeric column, all test ids
for f in sorted(glob.glob(INP + '/**/*.csv', recursive=True)):
    if '/playground-series-s6e10/' in f or 'oof' in os.path.basename(f).lower():
        continue
    try:
        head = pd.read_csv(f, nrows=3)
    except Exception:
        continue
    if 'id' not in head.columns or head.shape[1] != 2:
        continue
    try:
        df = pd.read_csv(f)
        col = [c for c in df.columns if c != 'id'][0]
        if len(df) != N or not pd.api.types.is_numeric_dtype(df[col]):
            continue
        v = df.set_index('id')[col].reindex(IDS).values.astype(float)
        if np.isnan(v).any():
            continue
        C['csv:' + short(f)] = v
    except Exception as e:
        print('skip', f, e)
# 2) cdeotte TFM test arrays (row order = test.csv)
for f in sorted(glob.glob(INP + '/**/test_*.npy', recursive=True)):
    v = np.load(f).astype(float).ravel()
    if len(v) == N and np.isfinite(v).all():
        C['npy:' + os.path.basename(f)[5:-4]] = v
# 3) my own model test predictions (aligned by saved ids)
for f in sorted(glob.glob(INP + '/**/s6e10_out/models/*/test.npy', recursive=True)):
    root = os.path.dirname(os.path.dirname(os.path.dirname(f)))
    ids = np.load(root + '/ids_test.npy', allow_pickle=True)
    v = pd.Series(np.load(f).astype(float), index=ids).reindex(IDS).values
    if not np.isnan(v).any():
        C['own:' + f.split('/')[-2]] = v

R = {k: rankdata(v) / N for k, v in C.items()}
names = list(C)
print(len(names), 'candidate vectors')

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def is_prob(v):
    return v.min() >= 0 and v.max() <= 1

# de-duplicate near-identical vectors (keep first)
uniq = []
for k in names:
    if all(np.corrcoef(R[k], R[u])[0, 1] < 0.999999 for u in uniq):
        uniq.append(k)
print(len(uniq), 'distinct after dedupe')

def find(pattern, pool=None):
    pool = pool or uniq
    hits = [k for k in pool if re.search(pattern, k, re.I)]
    return hits[0] if hits else None

ANCHOR = find(r'h-blend') or find(r'summit') or find(r'ensembling') or find(r'secrets')
print('anchor:', ANCHOR)
ra = R[ANCHOR]
tab = pd.DataFrame({'name': uniq,
                    'spearman_vs_anchor': [np.corrcoef(R[k], ra)[0, 1] for k in uniq],
                    'mean': [C[k].mean() for k in uniq]}).sort_values('spearman_vs_anchor', ascending=False)
pd.set_option('display.width', 200, 'display.max_colwidth', 80, 'display.max_rows', 200)
print(tab.to_string(index=False))

def blend(parts, space='logit'):
    tot = sum(w for _, w in parts)
    if space == 'rank':
        z = sum(w * R[k] for k, w in parts) / tot
        return z
    z = sum(w * logit(C[k] if is_prob(C[k]) else R[k]) for k, w in parts) / tot
    return 1 / (1 + np.exp(-z))

top = [k for k in uniq if np.corrcoef(R[k], ra)[0, 1] > 0.9990]
mega = find(r'megastack-plus-own/submission_cv_validated') or find(r'lr_stack_with_own')
tpfn = find(r'npy:tabpfn35')
kumo = find(r'npy:kumolarge')
mehran = find(r'1-airline-s6e10-tabpfn')
ownall = find(r'notebook12558cd5c0.*stack_all')
ownxgb = find(r'notebook12558cd5c0.*submission_xgb')
realmlp = find(r'part-c-realmlp.*submission_realmlp')
aysat = find(r'are-you-satisfied/submission\.csv')
print('top files:', top)
for nm, v in [('mega', mega), ('tabpfn35', tpfn), ('kumolarge', kumo), ('mehran', mehran), ('ownall', ownall),
              ('ownxgb', ownxgb), ('realmlp', realmlp), ('aysat', aysat)]:
    print(f'  {nm:10s} -> {v}')


meta = find(r'meta-stack-on-public-oof-lb-0-96213/submission\.csv')
summit = find(r'summit-composer-0-96210/submission\.csv')
xgbn = find(r'npy:xgboost')
rmlpn = find(r'npy:realmlp')
print('meta:', meta, '| summit:', summit, '| xgb npy:', xgbn, '| realmlp npy:', rmlpn)
recipes = {}
def add(tag, parts, space='rank'):
    if not parts or any(k is None for k, _ in parts):
        print('skip recipe', tag, '(missing component)')
        return
    recipes[tag] = (parts, space)

A = ANCHOR
add('r01_A95_tabpfn35', [(A, .95), (tpfn, .05)])
add('r02_A90_tabpfn35', [(A, .90), (tpfn, .10)])
add('r03_A95_mehran', [(A, .95), (mehran, .05)])
add('r04_A95_ownall', [(A, .95), (ownall, .05)])
add('r05_A90_mega', [(A, .90), (mega, .10)])
add('r06_A95_aysat', [(A, .95), (aysat, .05)])
add('r07_A90_aysat', [(A, .90), (aysat, .10)])
add('r08_A95_realmlp', [(A, .95), (realmlp, .05)])
add('r09_rankmean_top', [(k, 1) for k in top])
add('r10_A90_tabpfn35_ownall', [(A, .90), (tpfn, .05), (ownall, .05)])
add('r11_A85_tabpfn35_ownall_aysat', [(A, .85), (tpfn, .05), (ownall, .05), (aysat, .05)])
add('r12_A97_tabpfn35', [(A, .97), (tpfn, .03)])
add('r13_A90_mega_tabpfn35', [(A, .90), (mega, .05), (tpfn, .05)])
add('r14_A95_kumolarge', [(A, .95), (kumo, .05)])
add('r15_A95_ownxgb', [(A, .95), (ownxgb, .05)])
add('r16_A80_tabpfn35', [(A, .80), (tpfn, .20)])
# v3: anchor alone + negative-weight probes (public LB fell as these components were added)
add('n00_anchor', [(A, 1.0)])
add('n01_A105_tabpfn35_m05', [(A, 1.05), (tpfn, -.05)])
add('n02_A110_tabpfn35_m10', [(A, 1.10), (tpfn, -.10)])
add('n03_A105_mehran_m05', [(A, 1.05), (mehran, -.05)])
add('n04_A105_aysat_m05', [(A, 1.05), (aysat, -.05)])
add('n05_A110_tabpfn35_aysat_m05', [(A, 1.10), (tpfn, -.05), (aysat, -.05)])
add('n06_A105_ownall_m05', [(A, 1.05), (ownall, -.05)])
add('n07_A105_mega_m05', [(A, 1.05), (mega, -.05)])
add('n08_A115_tabpfn35_m15', [(A, 1.15), (tpfn, -.15)])
add('n09_A103_tabpfn35_m03', [(A, 1.03), (tpfn, -.03)])
# v4 (day 2): finer probes around the -5%..-10% TabPFN plateau + second negative component / other anchors
add('p01_A107_tpfn_m07', [(A, 1.07), (tpfn, -.07)])
add('p02_A108_tpfn_m05_kumo_m03', [(A, 1.08), (tpfn, -.05), (kumo, -.03)])
add('p03_A108_tpfn_m05_xgbn_m03', [(A, 1.08), (tpfn, -.05), (xgbn, -.03)])
add('p04_A108_tpfn_m05_rmlpn_m03', [(A, 1.08), (tpfn, -.05), (rmlpn, -.03)])
add('p05_A108_tpfn_m05_ownall_m03', [(A, 1.08), (tpfn, -.05), (ownall, -.03)])
add('p06_A108_tpfn_m05_mega_m03', [(A, 1.08), (tpfn, -.05), (mega, -.03)])
add('p07_A110_tpfn_m07_aysat_m03', [(A, 1.10), (tpfn, -.07), (aysat, -.03)])
add('p08_META105_tpfn_m05', [(meta, 1.05), (tpfn, -.05)])
add('p09_SUMMIT105_tpfn_m05', [(summit, 1.05), (tpfn, -.05)])
add('p10_A113_tpfn_m05_aysat_m05_kumo_m03', [(A, 1.13), (tpfn, -.05), (aysat, -.05), (kumo, -.03)])
# day 2 round 2: -3% deotte xgboost and -3% deotte realmlp each gave 0.96215 -> combine / scale
add('q01_A111_tpfn05_xgb03_rmlp03', [(A, 1.11), (tpfn, -.05), (xgbn, -.03), (rmlpn, -.03)])
add('q02_A110_tpfn05_xgb05', [(A, 1.10), (tpfn, -.05), (xgbn, -.05)])
add('q03_A110_tpfn05_rmlp05', [(A, 1.10), (tpfn, -.05), (rmlpn, -.05)])
add('q04_A114_tpfn05_xgb03_rmlp03_aysat03', [(A, 1.14), (tpfn, -.05), (xgbn, -.03), (rmlpn, -.03), (aysat, -.03)])
add('q05_A113_tpfn07_xgb03_rmlp03', [(A, 1.13), (tpfn, -.07), (xgbn, -.03), (rmlpn, -.03)])
add('q06_A115_tpfn05_xgb05_rmlp05', [(A, 1.15), (tpfn, -.05), (xgbn, -.05), (rmlpn, -.05)])
add('q07_A106_xgb03_rmlp03', [(A, 1.06), (xgbn, -.03), (rmlpn, -.03)])
# day 3: other Deotte TFMs on top of q07, xgb/rmlp scale, other anchors
tabfm = find(r'npy:tabfm'); limix = find(r'npy:limix2'); exaone = find(r'npy:exaone')
kumos = find(r'npy:kumosmall'); ticl = find(r'npy:tabicl2'); causilo = find(r'npy:causilo'); mitra = find(r'npy:mitra2')
print('day3 tfms:', tabfm, limix, exaone, kumos, ticl, causilo, mitra)
Q7 = [(xgbn, -.03), (rmlpn, -.03)]
add('t01_q07_tabfm_m02', [(A, 1.08)] + Q7 + [(tabfm, -.02)])
add('t02_q07_limix_m02', [(A, 1.08)] + Q7 + [(limix, -.02)])
add('t03_q07_exaone_m02', [(A, 1.08)] + Q7 + [(exaone, -.02)])
add('t04_q07_kumosmall_m02', [(A, 1.08)] + Q7 + [(kumos, -.02)])
add('t05_q07_tabicl_m02', [(A, 1.08)] + Q7 + [(ticl, -.02)])
add('t06_q07_causilo_m02', [(A, 1.08)] + Q7 + [(causilo, -.02)])
add('t07_q07_mitra_m02', [(A, 1.08)] + Q7 + [(mitra, -.02)])
add('t08_A104_xgb02_rmlp02', [(A, 1.04), (xgbn, -.02), (rmlpn, -.02)])
add('t09_A108_xgb04_rmlp04', [(A, 1.08), (xgbn, -.04), (rmlpn, -.04)])
add('t10_META106_xgb03_rmlp03', [(meta, 1.06)] + Q7)
add('t11_SUMMIT106_xgb03_rmlp03', [(summit, 1.06)] + Q7)
add('t12_q07_all7tfm_m01', [(A, 1.13)] + Q7 + [(v, -.01) for v in (tabfm, limix, exaone, kumos, ticl, causilo, mitra)])
# day 3 round 2: meta-stack anchor gave 0.96216 with -3% xgb -3% rmlp
M = meta
add('u01_META108_xgb04_rmlp04', [(M, 1.08), (xgbn, -.04), (rmlpn, -.04)])
add('u02_META104_xgb02_rmlp02', [(M, 1.04), (xgbn, -.02), (rmlpn, -.02)])
add('u03_META111_tpfn05_xgb03_rmlp03', [(M, 1.11), (tpfn, -.05), (xgbn, -.03), (rmlpn, -.03)])
add('u04_META109_xgb03_rmlp03_aysat03', [(M, 1.09), (xgbn, -.03), (rmlpn, -.03), (aysat, -.03)])
add('u05_META053_A053_xgb03_rmlp03', [(M, .53), (A, .53), (xgbn, -.03), (rmlpn, -.03)])
add('u06_META108_tpfn02_xgb03_rmlp03', [(M, 1.08), (tpfn, -.02), (xgbn, -.03), (rmlpn, -.03)])
add('u07_META105_xgb05', [(M, 1.05), (xgbn, -.05)])
add('u08_META105_rmlp05', [(M, 1.05), (rmlpn, -.05)])
# day 3 round 3: META - 5% xgboost alone = 0.96217
add('v01_META107_xgb07', [(M, 1.07), (xgbn, -.07)])
add('v02_META104_xgb04', [(M, 1.04), (xgbn, -.04)])
add('v03_META106_xgb06', [(M, 1.06), (xgbn, -.06)])
add('v04_META110_xgb10', [(M, 1.10), (xgbn, -.10)])
add('v05_META106_xgb05_rmlp01', [(M, 1.06), (xgbn, -.05), (rmlpn, -.01)])

rows = []
for tag, (parts, space) in recipes.items():
    p = blend(parts, space)
    pd.DataFrame({'id': IDS, 'satisfaction': p}).to_csv(f'{OUT}/{tag}.csv', index=False)
    rows.append({'file': tag + '.csv', 'spearman_vs_anchor': round(np.corrcoef(rankdata(p), ra)[0, 1], 7),
                 'parts': ' + '.join(f'{w}*{k.split("/")[-2] if "/" in k else k}' for k, w in parts)})
man = pd.DataFrame(rows)
man.to_csv(f'{OUT}/blend_manifest.csv', index=False)
print(man.to_string(index=False))
